# 03 · Modelo SVM

Objetivo: entrenar y ajustar un clasificador Support Vector Machine.

- Búsqueda de hiperparámetros (C, kernel, gamma)
- Validación cruzada
- Métricas de desempeño
- Persistencia del modelo entrenado

In [1]:

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    classification_report
)

# Rutas del proyecto
carpeta_datos = Path("../data/procesado")
carpeta_modelos = Path("../models")
carpeta_modelos.mkdir(parents=True, exist_ok=True)

# Cargar datos de entrenamiento y prueba
X_train = np.load(carpeta_datos / "X_train.npy")
X_test = np.load(carpeta_datos / "X_test.npy")
y_train = np.load(carpeta_datos / "y_train.npy")
y_test = np.load(carpeta_datos / "y_test.npy")

# Etiquetas que vamos a predecir
columnas_sabores = [
    "dulce", "salado", "amargo", "acido", "umami"
]

print("Datos cargados correctamente")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


Datos cargados correctamente
X_train: (3465, 1814)
X_test: (1486, 1814)
y_train: (3465, 5)
y_test: (1486, 5)


In [2]:

# Crear el modelo SVM multietiqueta
modelo_svm = Pipeline([
    ("escalador", StandardScaler()),
    ("clasificador", OneVsRestClassifier(
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced"
        ),
        n_jobs=-1
    ))
])

# Entrenar el modelo
print("Entrenando SVM...")
modelo_svm.fit(X_train, y_train)

# Guardar el modelo
joblib.dump(
    modelo_svm,
    carpeta_modelos / "svm_multietiqueta.joblib"
)

print("Modelo SVM entrenado y guardado.")


Entrenando SVM...
Modelo SVM entrenado y guardado.


In [3]:

# Predecir los sabores del conjunto de prueba
y_pred_svm = modelo_svm.predict(X_test)

# Calcular métricas generales
metricas_svm = {
    "F1 macro": f1_score(
        y_test, y_pred_svm, average="macro", zero_division=0
    ),
    "F1 micro": f1_score(
        y_test, y_pred_svm, average="micro", zero_division=0
    ),
    "Precision macro": precision_score(
        y_test, y_pred_svm, average="macro", zero_division=0
    ),
    "Recall macro": recall_score(
        y_test, y_pred_svm, average="macro", zero_division=0
    ),
    "Hamming loss": hamming_loss(y_test, y_pred_svm)
}

print("MÉTRICAS DEL MODELO SVM")
for nombre, valor in metricas_svm.items():
    print(f"{nombre}: {valor:.4f}")

print("\nREPORTE POR SABOR")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=columnas_sabores,
    zero_division=0
))


MÉTRICAS DEL MODELO SVM
F1 macro: 0.5008
F1 micro: 0.6800
Precision macro: 0.4941
Recall macro: 0.5088
Hamming loss: 0.2777

REPORTE POR SABOR
              precision    recall  f1-score   support

       dulce       0.53      0.50      0.51       571
      salado       0.81      0.89      0.84      1127
      amargo       0.00      0.00      0.00        67
       acido       0.42      0.40      0.41       509
       umami       0.72      0.76      0.74       927

   micro avg       0.68      0.68      0.68      3201
   macro avg       0.49      0.51      0.50      3201
weighted avg       0.65      0.68      0.67      3201
 samples avg       0.68      0.68      0.65      3201



In [5]:

# Guardar las predicciones
np.save(
    carpeta_datos / "y_pred_svm.npy",
    y_pred_svm
)

# Guardar las métricas
pd.DataFrame([
    {"modelo": "SVM", "metrica": nombre, "valor": valor}
    for nombre, valor in metricas_svm.items()
]).to_csv(
    carpeta_datos / "metricas_svm.csv",
    index=False
)

print("Predicciones y métricas guardadas correctamente.")


Predicciones y métricas guardadas correctamente.
